# **JATS Academy Webiners 第9回** #
# **AIを用いたデータ解析と論文作成 #2: 様々な機械学習モデルを用いたデータ解析**

日時: 2026年8月24日(月) 18:30-19:30


【ご注意】
*   本ノートブックは上記ウェビナーでのハンズオンのみを目的として作成されています。
*   本コードを実データや研究等に利用される場合は、利用者自身の責任において実施してください。
*   本コードの実行によって生じたいかなる不具合・損害についても、JATSならびに講演者は一切の責任を負いかねます。
*   本資料の転載・再配布はご遠慮ください。

# **1.サポートベクターマシン(SVM)**

  スライド ページ12: Python上でのSVMの実際

In [ ]:
# 解析に必要なモジュールの読み込み
import numpy as np
import pandas as pd

# 乳がんデータの読み込み
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer(as_frame=True)

# 分類に使用する特徴量と正解データの抽出
x_bc2 = bc.data.iloc[:,0:2]
y_bc = bc.target

# 学習データとテストデータにsplit
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_bc2, y_bc, test_size=0.2, random_state=100)

# 学習モデルとしてサポートベクターマシン(SVM)を選択する
from sklearn.svm import SVC
model_svm = SVC(kernel='rbf', random_state=0)   # 今回は非線形の分離境界を設定したいのでkernelは'rbf'を指定する

# 学習データを入れて学習させる
model_svm.fit(x_train, y_train)

スライド ページ13: Python上でのSVMの実際

In [ ]:
# テスト用データで予測値と正解値の比較
print(model_svm.predict(x_test))
print(np.array(y_test))

In [ ]:
# モデルの精度を正解率(Accuracy)で評価
print(model_svm.score(x_test, y_test))

スライド ページ14: SVMを用いたモデルが描く非線形境界

(以下のようなコードで描写することが可能です)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# グラフのサイズを設定
plt.figure(figsize=(10, 6))

# 各特徴量の最小値と最大値から、メッシュグリッド（描画領域）の範囲を決定
x_min, x_max = x_bc2.iloc[:, 0].min() - 1, x_bc2.iloc[:, 0].max() + 1
y_min, y_max = x_bc2.iloc[:, 1].min() - 1, x_bc2.iloc[:, 1].max() + 1

# 0.1刻みでメッシュグリッドを作成
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.1),
                     np.arange(y_min, y_max, 0.1))

# メッシュグリッド上のすべての点に対して予測を行う
grid_data = pd.DataFrame(np.c_[xx.ravel(), yy.ravel()], columns=x_bc2.columns)
Z = model_svm.predict(grid_data)
Z = Z.reshape(xx.shape)

# カラーマップの設定 (0:オレンジ色, 1:青色)
cmap_region = ListedColormap(['#ff7f0e', '#1f77b4'])

# 決定境界を等高線としてプロット
plt.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_region)

# テストデータの正解が0(malignant)のものをオレンジ色の三角印でプロット
plt.scatter(x_test.loc[y_test == 0].iloc[:, 0], x_test.loc[y_test == 0].iloc[:, 1],
            c='#ff7f0e', edgecolor='k', marker='^', label='Test Data (0: malignant)')

# テストデータの正解が1(benign)のものを青色の丸印でプロット
plt.scatter(x_test.loc[y_test == 1].iloc[:, 0], x_test.loc[y_test == 1].iloc[:, 1],
            c='#1f77b4', edgecolor='k', marker='o', label='Test Data (1: benign)')

plt.xlabel('mean radius')
plt.ylabel('mean texture')
plt.title('SVM Decision Boundary (Swapped Colors/Markers)')
plt.legend()
plt.show()

# **2. 決定木(DT)**

スライド ページ19: Python上での決定木の実際

In [ ]:
# 解析に必要なモジュールの読み込み。今回はプロットもするのでmatplotlibも入れる。
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 乳がんデータの読み込み
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer(as_frame=True)

# 分類に使用する特徴量と正解データの抽出
x_bc = bc.data     #今回は30特徴量すべてを使うので、.dataをそのままx_bcとする。
y_bc = bc.target

# 学習データとテストデータにsplit
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_bc, y_bc, test_size=0.2, random_state=100)

# 学習モデルとして決定木(DT)を選択する。この際にハイパーパラメータも一緒に設定する。
from sklearn.tree import DecisionTreeClassifier
model_dt = DecisionTreeClassifier(
    criterion='gini',         # 純度評価法はジニ不純度('gini')を選択
    max_depth=2,              # ルートノードからリーフノードまでの深さの最大値は2に設定
    min_samples_split=10,      # 最低サンプルサイズは10に設定
    random_state=0)

# 学習データを入れて学習させる
model_dt.fit(x_train, y_train)

スライド ページ20: Python上での決定機の実際

In [ ]:
# テスト用データで予測値と正解値の比較
print(model_dt.predict(x_test))
print(np.array(y_test))

In [ ]:
# モデルの精度を正解率(Accuracy)で評価
print(model_dt.score(x_test, y_test))

スライド ページ21: Python上での決定木の実際と図示

In [ ]:
from sklearn.tree import plot_tree
plot_tree(
    model_dt,
    feature_names=bc.feature_names,   # 特徴量の名前の一覧をNumPy配列で入力している
    class_names=bc.target_names,      # 正解値の分類名をNumPy配列で入力している
    filled=True,                      # Trueだとプロット図のノードに色をつける
    rounded=True,                     # ノードの角を丸めるかどうか
    fontsize=8,                       # プロット図のフォントの大きさを指定
)

# **3. ランダムフォレスト(DT)**

スライド ページ28: Python上でのRFの実際

In [ ]:
# 解析に必要なモジュールの読み込み。
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 乳がんデータの読み込み
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer(as_frame=True)

# 分類に使用する特徴量と正解データの抽出
x_bc = bc.data     # 今回も30特徴量すべてを使うので、.dataをそのままx_bcとする。
y_bc = bc.target

# 学習データとテストデータにsplit
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_bc, y_bc, test_size=0.2, random_state=100)

# 学習モデルとしてランダムフォレスト(RF)を選択する。ハイパーパラメータもとりあえず初期値を指定する。
from sklearn.ensemble import RandomForestClassifier
model_rf = RandomForestClassifier(
    n_estimators=100,    # 決定木の本数は100本
    max_depth=3,         # 木の深さ
    max_features=5,      # 各ノードの特徴量の数は5つ
    random_state=100
)

# 学習データを入れて学習させる
model_rf.fit(x_train, y_train)

スライド ページ29: Python上でのRFの実際

In [ ]:
# テスト用データで予測値と正解値の比較
print(model_rf.predict(x_test))
print(np.array(y_test))

In [ ]:
# モデルの精度を正解率(Accuracy)で評価
print(model_rf.score(x_test, y_test))

# **4. XGBoost**

スライド ページ35: Python上でのXGBoostの実際

In [ ]:
# 解析に必要なモジュールの読み込み。
import numpy as np
import pandas as pd
import xgboost as xgb   # xgboostのモジュール

# 乳がんデータの読み込み
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer(as_frame=True)

# 分類に使用する特徴量と正解データの抽出
x_bc = bc.data
y_bc = bc.target

# 学習データとテストデータにsplit
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_bc, y_bc, test_size=0.2, random_state=100)

# XGBoost分類モデルの定義
model_xgb = xgb.XGBClassifier(
    n_estimators=100,    # 木の本数
    max_depth=3,         # 木の深さ
    random_state=100,
    eval_metric='logloss'
)


スライド ページ36: Python上でのXGBoostの実際

In [ ]:
# 学習データを入れて学習させる
model_xgb.fit(x_train, y_train)

In [ ]:
# モデルの精度をテストデータの正解率(Accuracy)で評価
print(model_xgb.score(x_test, y_test))